# AGender - Step 2B: the leakage audit (CPU)

**Threat model.** The operator releases the 24 cleaned attributes of every subscriber. The partner already knows
age and gender of some subscribers (the training, validation and calibration parts of `splits/`) and wants them for
the others (the test part).

- **Attackers:** Logistic Regression, LightGBM, CatBoost, MLP and the baseline multi-task network;
  settings chosen on the validation part only.
- **Conformal sets:** built on the calibration part; for every test subscriber a set of possible age groups (and
  genders) that holds the true one with probability at least 90 % (also 80 % and 95 %); plain and class-conditional
  (the guarantee then holds inside each age group and each gender).
- **Per-subscriber leakage:** a single-class set means the subscriber is pinned down.
- **Copenhagen Networks Study:** the same audit for gender (582 students, raw call and SMS logs).
- Five repeats; 95 % bootstrap intervals on the test part.

**Kaggle settings:** Accelerator = **None** (CPU). Internet = Off is fine.
**Inputs to add:** `agender-code` and `agender-step1`.
**Expected run time:** about 5-10 minutes.


In [ ]:
import os
import sys
from pathlib import Path

# The AGender code is attached as a Kaggle dataset. Find the `agender` package
# wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('audit.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The agender-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> agender-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if package.parent.name == 'agender' and (package.parent / '__init__.py').exists():
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder; link the files under the name `agender`.
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'agender').exists():
        os.symlink(package.parent, code_root / 'agender')

sys.path.insert(0, str(code_root))
import agender
print('AGender code loaded from', Path(agender.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/agender_outputs/step2b')
TIME_BUDGET_MINUTES = 60
REPEATS = range(5)


In [ ]:
import json

import pandas as pd

from agender import environment, step2b
from agender.runner import StepRunner

runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {}
# Tables and splits come from the saved output of Step 1 (agender-step1).
files = {name: find_file(name, INPUT) for name in
         ('lat_clean.csv', 'lat_splits.csv', 'copenhagen_people.csv', 'copenhagen_splits.csv')}
print({k: str(v) for k, v in files.items()})
if None in files.values():
    raise RuntimeError('Attach the saved output of Step 1 (agender-step1): it holds lat_clean.csv, '
                       'lat_splits.csv, copenhagen_people.csv and copenhagen_splits.csv.')

summary['environment'] = runner.run('measure the machine', environment.collect)


In [ ]:
clean = pd.read_csv(files['lat_clean.csv'])
lat_splits = pd.read_csv(files['lat_splits.csv'])
summary['lat'] = runner.run('LAT audit (gender and six age groups)', step2b.audit_lat, clean, lat_splits, OUTPUT,
                            repeats=REPEATS)
for name, values in (summary['lat'] or {}).items():
    print(f'{name:28s}', values)


In [ ]:
people = pd.read_csv(files['copenhagen_people.csv'])
cns_splits = pd.read_csv(files['copenhagen_splits.csv'])
summary['copenhagen'] = runner.run('Copenhagen audit (gender)', step2b.audit_copenhagen, people, cns_splits, OUTPUT,
                                   repeats=REPEATS)
for name, values in (summary['copenhagen'] or {}).items():
    print(f'{name:28s}', values)


In [ ]:
summary['steps'] = runner.steps
with open(OUTPUT / 'step2b_summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=str)
print('Failed or skipped steps:', runner.failed() or 'none')
